# Mamba-PFAN Four-Way Training - 200 Epochs

This notebook trains the four-way Mamba-PFAN model for 200 epochs on the Kaggle GPU and evaluates it with the repository's standardized PSNR, SSIM, MSE, and CIEDE2000 protocol.

Before running:
1. Select a Kaggle GPU accelerator (T4/P100 or better).
2. Enable Internet so the notebook can clone `https://github.com/GladwinDaniel/Desmoking.git`.
3. Add the uploaded paired desmoking dataset under **+ Add Input**.

The notebook searches `/kaggle/input` recursively for a dataset containing `train` and `test` folders. It supports the aligned side-by-side format used by the repository and separate `trainA`/`trainB`, `testA`/`testB` folders.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import urllib.request
import zipfile

import torch

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/mamba_pfan_200_epochs')
CHECKPOINT_ROOT = WORK_ROOT / 'checkpoints'
RESULT_ROOT = WORK_ROOT / 'results'
CLONE_ROOT = WORK_ROOT / 'Desmoking'
# GitHub main is the standalone Mamba repository; its files are at the clone root.
REPO_ROOT = CLONE_ROOT
REPO_URL = 'https://github.com/GladwinDaniel/Desmoking.git'
REPO_BRANCH = 'main'

# Main training settings. Set RESUME to True after an interrupted Kaggle session.
EPOCHS = 200
BATCH_SIZE = 4
NUM_THREADS = 2
PRINT_FREQ = 20
SAVE_EPOCH_FREQ = 5
EXPERIMENT_NAME = 'mamba_pfan_four_way_200ep'
RESUME = False

GPU_IDS = '0' if torch.cuda.is_available() else '-1'
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: GPU not detected; training on CPU will be very slow.')

WORK_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Working directory: {WORK_ROOT}')
print(f'Internet available: {urllib.request.urlopen(REPO_URL, timeout=10).status == 200}')
print(f'Inputs: {[p.name for p in INPUT_ROOT.iterdir()]}')


In [ ]:
# Clone the current repository and install the Mamba-PFAN dependencies.
%pip install -q timm einops scikit-image Pillow numpy dominate

if CLONE_ROOT.exists():
    shutil.rmtree(CLONE_ROOT)
subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(CLONE_ROOT)],
    check=True,
)

required = ['train.py', 'evaluate_models.py', 'report_parameters.py', 'models', 'data']
missing = [name for name in required if not (REPO_ROOT / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing expected files in {REPO_ROOT}: {missing}')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
%cd {REPO_ROOT}
print(f'Repository: {REPO_ROOT}')
print(subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip())


In [ ]:
# Locate the uploaded dataset and normalize separate A/B folders for training.
from PIL import Image

VALID_EXTENSIONS = {'.bmp', '.png', '.jpg', '.jpeg'}

def has_images(folder):
    return folder.is_dir() and any(p.suffix.lower() in VALID_EXTENSIONS for p in folder.iterdir())

def find_dataset(root):
    for current, dirs, _ in os.walk(root):
        path = Path(current)
        if (path / 'train').is_dir() and has_images(path / 'train'):
            test = path / 'test'
            if test.is_dir():
                return path, test
        if (path / 'trainA').is_dir() and (path / 'trainB').is_dir():
            test = path
            if (path / 'testA').is_dir() and (path / 'testB').is_dir():
                return path, test
    return None, None

dataset_root, test_root = find_dataset(INPUT_ROOT)
if dataset_root is None:
    archives = [p for p in INPUT_ROOT.rglob('*.zip') if 'mamba' not in p.name.lower()]
    if archives:
        extracted = WORK_ROOT / 'extracted_dataset'
        extracted.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archives[0]) as archive:
            archive.extractall(extracted)
        dataset_root, test_root = find_dataset(extracted)

if dataset_root is None:
    raise FileNotFoundError(
        'No dataset found. Add an input containing train/test or trainA/trainB and testA/testB.'
    )

DATA_ROOT = dataset_root
TEST_ROOT = test_root
if (DATA_ROOT / 'trainA').is_dir() and (DATA_ROOT / 'trainB').is_dir():
    aligned_root = WORK_ROOT / 'aligned_dataset'
    aligned_train = aligned_root / 'train'
    aligned_train.mkdir(parents=True, exist_ok=True)
    a_files = sorted(p for p in (DATA_ROOT / 'trainA').iterdir() if p.suffix.lower() in VALID_EXTENSIONS)
    created = 0
    for a_path in a_files:
        b_path = DATA_ROOT / 'trainB' / a_path.name
        if not b_path.is_file():
            continue
        image_a = Image.open(a_path).convert('RGB')
        image_b = Image.open(b_path).convert('RGB')
        if image_a.size != image_b.size:
            raise ValueError(f'Paired image sizes differ: {a_path.name}')
        combined = Image.new('RGB', (image_a.width + image_b.width, image_a.height))
        combined.paste(image_a, (0, 0))
        combined.paste(image_b, (image_a.width, 0))
        combined.save(aligned_train / a_path.name)
        created += 1
    if created == 0:
        raise RuntimeError('No matching trainA/trainB image pairs were found.')
    DATA_ROOT = aligned_root

train_count = len([p for p in (DATA_ROOT / 'train').iterdir() if p.suffix.lower() in VALID_EXTENSIONS])
print(f'Dataset root for training: {DATA_ROOT}')
print(f'Test root for evaluation: {TEST_ROOT}')
print(f'Training pairs: {train_count}')


## Parameter report

This confirms the exact generator sizes used by the cloned code before training.

In [ ]:
PARAMETER_REPORT = WORK_ROOT / 'parameter_report.json'
subprocess.run(
    [sys.executable, str(REPO_ROOT / 'report_parameters.py'), '--output', str(PARAMETER_REPORT)],
    cwd=REPO_ROOT,
    check=True,
)
print(PARAMETER_REPORT.read_text())


## Train four-way Mamba-PFAN for 200 epochs

The command uses the same `train.py` entry point as the previous ablation notebook, but trains only `four_way`. Checkpoints are saved every five epochs. If Kaggle disconnects, set `RESUME = True` in the configuration cell and rerun this cell; the latest checkpoint will be continued.

In [ ]:
TRAIN_COMMAND = [
    sys.executable, str(REPO_ROOT / 'train.py'),
    '--dataroot', str(DATA_ROOT),
    '--name', EXPERIMENT_NAME,
    '--model', 'pix2pix',
    '--netG', 'mamba_pfan',
    '--netD', 'basic',
    '--direction', 'AtoB',
    '--dataset_mode', 'aligned',
    '--norm', 'batch',
    '--checkpoints_dir', str(CHECKPOINT_ROOT),
    '--embed_dim', '64',
    '--ndf', '64',
    '--ngf', '64',
    '--batch_size', str(BATCH_SIZE),
    '--num_threads', str(NUM_THREADS),
    '--n_epochs', str(EPOCHS),
    '--n_epochs_decay', '0',
    '--save_epoch_freq', str(SAVE_EPOCH_FREQ),
    '--print_freq', str(PRINT_FREQ),
    '--display_id', '-1',
    '--gpu_ids', GPU_IDS,
    '--scan_mode', 'four_way',
    '--no_flip',
]
if RESUME:
    TRAIN_COMMAND.append('--continue_train')

checkpoint = CHECKPOINT_ROOT / EXPERIMENT_NAME / 'latest_net_G.pth'
if RESUME and not checkpoint.is_file():
    raise FileNotFoundError(f'Cannot resume; checkpoint does not exist: {checkpoint}')
print('Running:', ' '.join(TRAIN_COMMAND))
subprocess.run(TRAIN_COMMAND, cwd=REPO_ROOT, check=True)
print(f'Final generator checkpoint: {checkpoint}')


## Evaluate on the untouched test split

This uses `evaluate_models.py`, the same implementation used for the existing 103-image ablation summaries.

In [ ]:
import json

FINAL_CHECKPOINT = CHECKPOINT_ROOT / EXPERIMENT_NAME / 'latest_net_G.pth'
EVAL_DIR = RESULT_ROOT / 'four_way_200_epochs'
EVAL_DIR.mkdir(parents=True, exist_ok=True)
if not FINAL_CHECKPOINT.is_file():
    raise FileNotFoundError(FINAL_CHECKPOINT)

EVAL_COMMAND = [
    sys.executable, str(REPO_ROOT / 'evaluate_models.py'),
    '--model', 'mamba_pfan',
    '--scan-mode', 'four_way',
    '--checkpoint', str(FINAL_CHECKPOINT),
    '--test-dir', str(TEST_ROOT),
    '--output-dir', str(EVAL_DIR),
    '--image-size', '256',
    '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
]
subprocess.run(EVAL_COMMAND, cwd=REPO_ROOT, check=True)
summary = json.loads((EVAL_DIR / 'evaluation_summary.json').read_text())
print(json.dumps(summary, indent=2))


In [ ]:
# Export metrics, options, logs, and the final generator checkpoint for download.
export_dir = Path('/kaggle/working/mamba_pfan_200_epochs_export')
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True)
shutil.copytree(EVAL_DIR, export_dir / 'evaluation')
shutil.copy(PARAMETER_REPORT, export_dir / PARAMETER_REPORT.name)
shutil.copy(FINAL_CHECKPOINT, export_dir / FINAL_CHECKPOINT.name)
options_file = CHECKPOINT_ROOT / EXPERIMENT_NAME / 'train_opt.txt'
if options_file.is_file():
    shutil.copy(options_file, export_dir / options_file.name)
loss_file = CHECKPOINT_ROOT / EXPERIMENT_NAME / 'loss_log.txt'
if loss_file.is_file():
    shutil.copy(loss_file, export_dir / loss_file.name)
print(f'Export bundle ready: {export_dir}')
for path in sorted(export_dir.rglob('*')):
    if path.is_file():
        print(path.relative_to(export_dir), path.stat().st_size, 'bytes')
